In [10]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks

# Obtener la ruta del directorio raíz del proyecto
project_root = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))

# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(project_root)

from config import INPUTS_FOLDER

statement_path = os.path.join(INPUTS_FOLDER, 'test_files', 'Banorte', 'Banorte_credit_new_statement.pdf')
statement_path

'c:\\Proyectos\\Aether\\Aether_v1\\documents\\inputs\\test_files\\Banorte\\Banorte_credit_new_statement.pdf'

In [11]:
from document_reader import PDFReader
from bank_detector import DefaultBankDetector

bank_detector = DefaultBankDetector(PDFReader(statement_path))

extracted_words = bank_detector.extracted_words
extracted_words

,page,text,x0,top,x1,bottom
0,2,2,553.038000,52.095053,557.486139,60.095303
1,2,de,559.703069,52.095053,568.599347,60.095303
2,2,8,570.831000,52.095053,575.279139,60.095303
3,2,MARIA,45.021750,72.287250,73.497750,81.287250
4,2,FERNANDA,75.990750,72.287250,125.454750,81.287250
...,...,...,...,...,...,...
4218,8,opinión,131.473888,754.353053,171.118789,765.353303
4219,8,de,174.165858,754.353053,186.992150,765.353303
4220,8,este,190.039219,754.353053,212.050719,765.353303
4221,8,comparativo:,215.097788,754.353053,284.124357,765.353303


In [12]:
bank = bank_detector.detect_bank()
statement_type = bank_detector.detect_statement_type()
statement_properties = bank_detector.get_statement_properties()

print(f'{bank} - {statement_type}')

banorte - credit


In [13]:
from table_boundary_detector import TransactionTableBoundaryDetector

boundary_detector = TransactionTableBoundaryDetector(extracted_words, statement_properties)

start_idx = boundary_detector.start_idx
end_idx = boundary_detector.end_idx

if statement_type == 'credit' and start_idx is None or end_idx is None:
    print('New format')
    statement_properties = bank_detector.get_statement_properties(new_credit_format= True)
    boundary_detector = TransactionTableBoundaryDetector(extracted_words, statement_properties)
    
    start_idx = boundary_detector.start_idx
    end_idx = boundary_detector.end_idx

df_table = boundary_detector.get_filtered_table_words()

print(f'{start_idx} - {end_idx}')
df_table

New format
1035 - 1279


,page,text,x0,top,x1,bottom
0,4,(NO,375.81825,142.42575,392.30625,151.42575
1,4,A,394.79925,142.42575,401.29725,151.42575
2,4,MESES),403.79025,142.42575,438.26025,151.42575
3,4,Tarjeta,224.30850,152.56125,253.13550,161.56125
4,4,titular,255.62850,152.56125,280.60350,161.56125
...,...,...,...,...,...,...
239,4,09/11/24,138.66450,580.09725,173.00850,589.09725
240,4,85.8515,175.50150,580.09725,208.01850,589.09725
241,4,USD,210.51150,580.09725,229.50150,589.09725
242,4,RT,231.99450,580.09725,243.82050,589.09725


In [14]:
from row_segmenter import TransactionRowSegmenter

row_segmenter = TransactionRowSegmenter(df_table, statement_properties)

column_delimitation = row_segmenter.delimit_column_positions()
grouped_rows = row_segmenter.group_rows()

print(row_segmenter.row_threshold)
print(column_delimitation)
grouped_rows

0.9954999999999989
{'columns': ['Fecha de la operación', 'Fecha de cargo', 'Descripción del movimiento', 'Monto'], 'x0': [36.816, 92.16825, 279.63525, 542.154], 'x1': [75.2703615, 118.84887524999999, 377.4143055, 564.3786944999999]}


,row_group,text,words,top,bottom,page
0,0,(NO A MESES),"[((NO, 375.8182499999999, 392.3062499999999), ...",142.425750,151.425750,4
1,1,Tarjeta titular XXXX XXXX XXXX 4540,"[(Tarjeta, 224.30849999999998, 253.13549999999...",152.561250,161.561250,4
2,2,Fecha de Fecha de,"[(Fecha, 36.816, 59.0406945), (de, 61.25676375...",165.994553,173.994802,4
3,3,Monto,"[(Monto, 542.154, 564.3786944999999)]",168.999053,176.999302,4
4,4,Descripción del movimiento,"[(Descripción, 279.63525, 321.42055575), (del,...",170.499053,178.499302,4
5,5,la operación cargo,"[(la, 31.701, 37.925194499999996), (operación,...",175.003553,183.003803,4
6,6,D LOCAL*DIDIFOOD CIUDAD DE MEX MX DMM 171208LT...,"[(D, 138.6645, 145.1625), (LOCAL*DIDIFOOD, 147...",188.781750,197.781750,4
7,7,AMAZON MX A MESES M CIUDAD DE M 02/03 12-SEP-2...,"[(AMAZON, 138.6645, 177.13049999999998), (MX, ...",205.059000,214.059000,4
8,8,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4 ...,"[(DLOCAL*DIDI, 138.6645, 196.13850000000002), ...",221.337000,230.337000,4
9,9,SIMILARES F3 OTE MERIDA YUC MX CCA 010830V77 1...,"[(SIMILARES, 138.6645, 186.62550000000002), (F...",237.615000,246.615000,4


In [15]:
from table_reconstructor import TransactionTableReconstructor

table_reconstructor = TransactionTableReconstructor(grouped_rows, column_delimitation, statement_properties)

table_reconstructor.get_structured_table()

,Fecha de la operación,Fecha de cargo,Descripción del movimiento,Monto
0,,,(NO A MESES),A MESES)
1,,,Tarjeta titular XXXX XXXX XXXX 4540,
2,Fecha de,Fecha de,,
3,,,,Monto
4,,,Descripción del movimiento,
5,la operación,cargo,,
6,10-SEP-2024,11-SEP-2024,D LOCAL*DIDIFOOD CIUDAD DE MEX MX DMM 171208LT4,+$426.24
7,12-SEP-2024,04-OCT-2024,AMAZON MX A MESES M CIUDAD DE M 02/03,"+$1,390.85"
8,13-SEP-2024,17-SEP-2024,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,+$56.00
9,17-SEP-2024,18-SEP-2024,SIMILARES F3 OTE MERIDA YUC MX CCA 010830V77,+$169.00


In [16]:
df_structured = table_reconstructor.reconstruct_table()
df_structured

,Fecha de la operación,Fecha de cargo,Descripción del movimiento,Monto
0,10-SEP-2024,11-SEP-2024,D LOCAL*DIDIFOOD CIUDAD DE MEX MX DMM 171208LT4,+$426.24
1,12-SEP-2024,04-OCT-2024,AMAZON MX A MESES M CIUDAD DE M 02/03,"+$1,390.85"
2,13-SEP-2024,17-SEP-2024,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,+$56.00
3,17-SEP-2024,18-SEP-2024,SIMILARES F3 OTE MERIDA YUC MX CCA 010830V77,+$169.00
4,21-SEP-2024,23-SEP-2024,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,+$84.00
5,21-SEP-2024,23-SEP-2024,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,+$6.00
6,21-SEP-2024,23-SEP-2024,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,+$23.25
7,21-SEP-2024,23-SEP-2024,"PAGO BANCA DIGITAL / SUCURSAL, GRACIAS.","-$5,630.57"
8,24-SEP-2024,25-SEP-2024,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,+$95.00
9,25-SEP-2024,26-SEP-2024,APPLE.COM/BILL 866-712-77,+$49.25


In [17]:
from table_normalizer import TransactionTableNormalizer

normalizer = TransactionTableNormalizer(df_structured, extracted_words, statement_properties)

df_normalized = normalizer.normalize_table()
df_normalized

,Date,Description,Amount,Type
0,2024-09-10,D LOCAL*DIDIFOOD CIUDAD DE MEX MX DMM 171208LT4,-426.24,Cargo
1,2024-09-12,AMAZON MX A MESES M CIUDAD DE M 02/03,-1390.85,Cargo
2,2024-09-13,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,-56.00,Cargo
3,2024-09-17,SIMILARES F3 OTE MERIDA YUC MX CCA 010830V77,-169.00,Cargo
4,2024-09-21,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,-84.00,Cargo
5,2024-09-21,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,-6.00,Cargo
6,2024-09-21,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,-23.25,Cargo
7,2024-09-21,"PAGO BANCA DIGITAL / SUCURSAL, GRACIAS.",5630.57,Abono
8,2024-09-24,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,-95.00,Cargo
9,2024-09-25,APPLE.COM/BILL 866-712-77,-49.25,Cargo


In [18]:
from data_exporter import CsvExporter
from config import OUTPUTS_FOLDER

exporter = CsvExporter(df_normalized)
file_path = os.path.join(OUTPUTS_FOLDER, bank, f'{bank}_{statement_type}.csv')

exporter.export_to_csv(file_path)

Data exported to c:\Proyectos\Aether\Aether_v1\documents\outputs\banorte\banorte_credit.csv successfully.
